### Tasks

layman terms: Tasks are wrappers around a coroutine that schedule a coroutine to run on the event loop as soon as possible.

I'm assuming tasks abstract away all the scheduling and execution such that maximum possible concurrency is achieved, as can be achieved by a single thread.

In [ ]:
# creating a task.

import asyncio
from util import delay
import time

async def main():
    sleep_for_three = asyncio.create_task(delay(3)) # creating the task.. VERY IMP: the task is ready for execution as soon as we create / schedule it using this syntax... the event loop will pick it as soon as its free... The await command down the line for this task is the same idea as joining your threads to make sure the thread has finished its job until that point...
    print(type(sleep_for_three))
    await asyncio.sleep(3)

    print("let's see how long the await command takes now...")
    s = time.time()
    result = await sleep_for_three # the task may have already completed before we awaited it... but we should await it manually at some point because it's good practice...
    e = time.time()

    print(e - s) # should read in micro-seconds... the task had enough time to finish on its own before we decided to manually await it...
    print(result)
    
asyncio.run(main())

# NOTE: the above is a good example on how you can tasks to get things done without blocking your containing coroutine function... the code will continue executing under it until you yourself decide (with the await command) that you do not want to go any further UNTIL that task is completed surely.

Another benefit of tasks is that you can schedule several in a row and expect that sweet concurrency you'd always wanted.

In [ ]:
import asyncio
from util import delay
import time

async def main():
    sleep_for_three = asyncio.create_task(delay(3))
    sleep_again = asyncio.create_task(delay(3))
    sleep_once_more = asyncio.create_task(delay(3))
    
    s = time.time()
    await sleep_for_three
    await sleep_again
    await sleep_once_more
    e = time.time()
    print(e - s)

asyncio.run(main())

# notice how you only waited 3s, even though the total was supposed to be 9s.

# NOTE: Very imp: you realise that right up until the first await in this codeblock, the event loop was probably busy with running this coroutine function itself? Only after that first await does event loop get time to execute other tasks that have been scheduled...

# remember this... Also rem that every await command is in itself a new iteration of the event loop... a break from execution that lets it check what other items from the menu / to-do list can be checked out.


# remember those gantt-chart like diagrams in the textbook... always remember, no green blocks should ever overlap.

### Cancelling tasks & specifying timeouts

We have control over tasks which may be running too long... or tasks that were misfired... we can cancel / have timeouts in place for these so the user doesn't have to wait forever.

In [ ]:
# Cancellation

import asyncio
from asyncio import CancelledError
from util import delay

async def main():
    long_task = asyncio.create_task(delay(10))
    seconds_elapsed = 0
    while not long_task.done():
        print('Task not finished, checking again in a second.')
        await asyncio.sleep(1)

        seconds_elapsed = seconds_elapsed + 1
        if seconds_elapsed == 5:
            long_task.cancel()
    try:
        await long_task
    except CancelledError:
        print('Our task was cancelled')

asyncio.run(main())

# NOTE: Task Cancellation Flow in asyncio
#
# 1. task.cancel() is called.
#    - This does NOT stop anything immediately.
#    - It just schedules the task to be resumed "as soon as possible"
#      (jumping ahead of its original await, e.g. sleep(10)), with a
#      CancelledError ready to be thrown into it.
#
# 2. Nothing changes synchronously. task.done() and task.cancelled()
#    are still False right after .cancel() is called.
#
# 3. Control must return to the event loop before anything happens.
#    (In our example, this occurs when main()'s polling loop hits
#    `await asyncio.sleep(1)` and yields control.)
#
# 4. The event loop, now free, sees long_task is ready to resume
#    (due to the cancellation request) and resumes it by THROWING
#    CancelledError in at the exact point it was suspended
#    (e.g. inside delay(), at `await asyncio.sleep(10)`).
#
# 5. Since delay() has no try/except around that await, it does NOT
#    continue running "as usual." The exception immediately propagates
#    out, uncaught, terminating the coroutine right there.
#
#    -> It is this uncaught propagation that CAUSES the task to end
#       in the cancelled state. (If delay() had caught the error and
#       returned normally instead, the task would finish as
#       done()=True, cancelled()=False — NOT cancelled.)
#
# 6. As a direct result of step 5, long_task.done() and
#    long_task.cancelled() both become True, at the same moment.
#    This all happens almost instantly - well within main()'s
#    1-second sleep, not after the original 10s would've elapsed.
#
# 7. main()'s sleep(1) finishes, control returns to the while loop.
#    `not long_task.done()` is now False -> loop exits.
#
# 8. `await long_task` is called. Since the task already finished via
#    an uncaught CancelledError, awaiting it RE-RAISES that same
#    CancelledError to the awaiter (main()), which is caught here
#    by the try/except.
#
# KEY IDEA: task.cancel() only requests cancellation. The task's
# done/cancelled status only updates once the event loop actually
# resumes that task's own coroutine and the injected CancelledError
# propagates out of it uncaught. Nothing is instant except the
# scheduling of the request itself.

### A gotcha about cancellation..

you may cancel a task at any point, but that cancellation won't be effectively immediately... 

What actually happens is that when you cancel a task, only the cancelled flag is set to true for the task, and the event loop can check this flag only when it has control... the event loop may then inject the 'CanceledError' status when it gets control back from the task... This means it must wait for the cancelled task to relinquish control to it on the next 'await'. It could be an await for anything... what matters is that after that await, event loop gets control, sees that this task is cancelled, and then you get a cancelledError which i'm assuming gets propagated... which you can then handle in exceptions.

a funny thing that's apparently used in practice..

```python
await asyncio.sleep(0)
```

You run this command at regular intervals in your code for long-running CPU bound tasks. This hands the event loop control every once in a while.. It's harmless, but it ensures that your task doesn't run for very long after it's been cancelled.

In [ ]:
# setting timeout & cancelling task after timeout

import asyncio
from util import delay

async def main():
    delay_task = asyncio.create_task(delay(2))
    try:
        result = await asyncio.wait_for(delay_task, timeout=1) # how to specify a timeout on a task: you await it with special syntax.
        print(result)
    except asyncio.exceptions.TimeoutError:
        print('Got a timeout!')
        print(f'Was the task cancelled? {delay_task.cancelled()}')

        
asyncio.run(main())

In [ ]:
# setting timeout & only raising exception after timeout (shielding)

# sometimes, we may want to set a timeout for a particular coroutine, but not outright cancel it once it passes the timeout.. You may instead want to give an update to the user saying 'this is taking longer than expected'.. This is where timeout with 'shield()' is useful. You shield the task from cancellation inspite of the timeout.

import asyncio
from util import delay

async def main():
    task = asyncio.create_task(delay(10))
    try:
        result = await asyncio.wait_for(asyncio.shield(task), 5)
        print(result)
    except TimeoutError:
        print("Task took longer than five seconds, it will finish soon!")
        result = await task
        print(result)

asyncio.run(main())

Next, we're going to learn about how tasks, coroutines are related.. This involves us understanding 2 other concepts - futures, and awaitables